# Projet NOZ - Étape 6 : prévoir le CA des 7 prochains jours

David DOE · 2026

La question métier : chaque lundi, le manager veut savoir combien le magasin va vendre sur les 7 prochains jours, pour prévoir l'effectif en caisse et en rayon (*capacity planning*).

Ce qu'on prévoit : le CA total du magasin par jour. On a choisi le CA plutôt que le stock, parce que le stock dépend des livraisons imposées par le siège (modèle *push*), que le magasin ne connaît pas à l'avance.

Le plan :
1. Regarder la série : tendance et saisonnalité.
2. Trois méthodes, de la plus simple à la plus élaborée.
3. Les tester comme dans la vraie vie : chaque lundi, on prévoit la semaine avec seulement le passé (validation glissante).
4. Comparer avec l'objectif du cahier des charges : une erreur au moins 15 % plus faible que la méthode naïve.

 Données simulées.

Installation (une seule fois) :
```
python -m pip install statsmodels
```

## 1. Connexion et vocabulaire

- Série temporelle : une suite de valeurs dans l'ordre du temps (ici, 1 CA par jour pendant 599 jours).
- Saisonnalité : un motif qui se répète. Ici, la semaine : samedi fort, dimanche faible.
- Tendance : la direction générale sur le long terme (ça monte, ça baisse ou c'est stable).
- Horizon : jusqu'où on prévoit (ici, 7 jours).

In [ ]:
import os, getpass, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sqlalchemy import create_engine, URL
from statsmodels.tsa.holtwinters import ExponentialSmoothing
warnings.filterwarnings("ignore")   # masque des avertissements techniques sans importance

url = os.environ.get("NOZ_DB_URL")
if url is None:
    url = URL.create("postgresql+psycopg2", username="postgres",
                     password=getpass.getpass("Mot de passe PostgreSQL : "),
                     host="localhost", port=5432, database="noz")
moteur = create_engine(url)
os.makedirs("figures", exist_ok=True)

jours = pd.read_sql("SELECT date_jour, jour_semaine, evenement, ca_total_eur::float AS ca "
                    "FROM magasin_jour ORDER BY date_jour", moteur, parse_dates=["date_jour"])
ca = jours.set_index("date_jour")["ca"].asfreq("D")   # "D" = une valeur par jour
print(f"{len(ca)} jours, du {ca.index.min():%d/%m/%Y} au {ca.index.max():%d/%m/%Y}")

## 2. Regarder la série avant de prévoir

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(11, 6))
ca.plot(ax=axes[0], color="#bdc3c7", linewidth=0.8, label="CA du jour")
ca.rolling(28, center=True).mean().plot(ax=axes[0], color="#c0392b", linewidth=2,
                                        label="moyenne sur 4 semaines (tendance)")
axes[0].set_title("CA quotidien du magasin (donnees simulees)")
axes[0].set_ylabel("EUR")
axes[0].legend()

zoom = ca["2025-03-03":"2025-03-30"]
axes[1].bar(zoom.index, zoom.values, color=["#c0392b" if d.dayofweek == 5 else "#7f8c8d" for d in zoom.index])
axes[1].set_title("Zoom sur 4 semaines : le motif hebdomadaire (samedi en rouge)")
axes[1].set_ylabel("EUR")
plt.tight_layout()
plt.savefig("figures/m6_serie_ca.png", dpi=150)
plt.show()

Ce qu'on voit :
- Un motif qui se répète chaque semaine (le samedi dépasse toujours). C'est la saisonnalité, et c'est elle qui rend la prévision possible.
- Des variations plus lentes au fil des mois (événements comme les Soldes ou Noël, été plus calme).
- Du bruit : deux samedis ne sont jamais identiques. Aucune méthode ne pourra le prévoir.

## 3. Trois méthodes de prévision

1. Naïve saisonnière (la référence) : « lundi prochain = lundi dernier ». C'est ce qu'un manager ferait de tête. Toute méthode sérieuse doit faire mieux qu'elle, sinon elle ne sert à rien.
2. Moyenne des 4 dernières semaines : « lundi prochain = moyenne des 4 derniers lundis ». Elle lisse le bruit d'une semaine exceptionnelle.
3. Holt-Winters (lissage exponentiel) : il apprend le niveau du moment et le profil de la semaine, en donnant plus de poids aux jours récents. C'est une méthode classique en commerce.
   - *multiplicatif* : le samedi vaut « +35 % » plutôt que « +2 800 € », donc l'effet suit le niveau des ventes.

In [ ]:
def prevoir(historique, methode):
    """Prevoit les 7 jours qui suivent la fin de l'historique."""
    if methode == "Naive saisonniere":
        return historique.iloc[-7:].to_numpy()
    if methode == "Moyenne 4 semaines":
        return historique.iloc[-28:].to_numpy().reshape(4, 7).mean(axis=0)
    if methode == "Holt-Winters":
        modele = ExponentialSmoothing(historique, trend=None,
                                      seasonal="mul", seasonal_periods=7).fit()
        return modele.forecast(7).to_numpy()

METHODES = ["Naive saisonniere", "Moyenne 4 semaines", "Holt-Winters"]

## 4. Tester comme dans la vraie vie : la validation glissante

On se met dans la peau du manager, chaque lundi de 2025 :
1. on ne connaît que le passé (tout ce qui précède ce lundi) ;
2. on prévoit les 7 jours de la semaine ;
3. la semaine se passe, et on mesure l'erreur ;
4. on avance d'une semaine et on recommence.

On obtient ainsi 32 semaines de test, au lieu d'un seul essai qui pourrait tomber par chance sur une semaine facile.

Les deux mesures d'erreur :
- MAE (erreur absolue moyenne) : de combien d'euros on se trompe en moyenne par jour.
- MAPE : la même erreur en pourcentage du CA du jour. C'est plus parlant : « on se trompe de 11 % ».

In [ ]:
lundis = pd.date_range("2025-01-06", ca.index.max() - pd.Timedelta(days=6), freq="7D")
lignes = []
for lundi in lundis:
    passe  = ca[: lundi - pd.Timedelta(days=1)]           # tout ce qu'on sait ce lundi matin
    semaine = ca[lundi : lundi + pd.Timedelta(days=6)]    # ce qui va se passer
    for m in METHODES:
        prev = prevoir(passe, m)
        for jour, reel, p in zip(semaine.index, semaine.to_numpy(), prev):
            lignes.append({"lundi": lundi, "date_jour": jour, "methode": m,
                           "reel": reel, "prevision": p})

resultats = pd.DataFrame(lignes)
resultats["erreur_abs"] = (resultats["reel"] - resultats["prevision"]).abs()
print(f"{len(lundis)} semaines testees")

In [ ]:
tableau = (resultats.groupby("methode")
           .agg(MAE_eur=("erreur_abs", "mean"),
                MAPE_pct=("erreur_abs", lambda e: 100 * (e / resultats.loc[e.index, "reel"]).mean()))
           .reindex(METHODES).round(1))
mae_naive = tableau.loc["Naive saisonniere", "MAE_eur"]
tableau["gain_vs_naive_pct"] = (100 * (1 - tableau["MAE_eur"] / mae_naive)).round(1)
tableau["objectif_15pct"] = np.where(tableau["gain_vs_naive_pct"] >= 15, "atteint", "non atteint")
tableau.loc["Naive saisonniere", "objectif_15pct"] = "(reference)"
tableau

Comment lire ce tableau :
- La méthode naïve se trompe d'environ 1 000 € par jour (≈ 14 %). C'est le niveau à battre.
- La moyenne sur 4 semaines fait mieux : lisser le bruit aide.
- Holt-Winters réduit l'erreur d'environ 20 % par rapport à la méthode naïve : l'objectif de 15 % est atteint.
- Mais il reste ≈ 11 % d'erreur : c'est le bruit du quotidien, imprévisible. Une prévision n'est jamais exacte, elle donne un ordre de grandeur fiable.

## 5. Voir les prévisions à côté de la réalité

In [ ]:
fin = resultats[resultats["lundi"] >= resultats["lundi"].max() - pd.Timedelta(weeks=7)]
fig, ax = plt.subplots(figsize=(11, 4.5))
reel = fin[fin["methode"] == "Holt-Winters"].set_index("date_jour")["reel"]
ax.plot(reel.index, reel.values, color="black", linewidth=2, label="CA reel")
for m, couleur in [("Naive saisonniere", "#95a5a6"), ("Holt-Winters", "#c0392b")]:
    p = fin[fin["methode"] == m].set_index("date_jour")["prevision"]
    ax.plot(p.index, p.values, color=couleur, linestyle="--", label=f"prevision {m}")
ax.set_ylabel("EUR")
ax.set_title("8 dernieres semaines : prevision chaque lundi pour les 7 jours suivants (donnees simulees)")
ax.legend()
plt.tight_layout()
plt.savefig("figures/m6_prevision_vs_reel.png", dpi=150)
plt.show()

## 6. Où se trompe-t-on le plus ?
On regarde l'erreur moyenne (en %) par jour de la semaine pour Holt-Winters.

In [ ]:
hw = resultats[resultats["methode"] == "Holt-Winters"].copy()
hw["erreur_pct"] = 100 * hw["erreur_abs"] / hw["reel"]
hw["jour"] = hw["date_jour"].dt.dayofweek.map(dict(enumerate(
    ["Lundi", "Mardi", "Mercredi", "Jeudi", "Vendredi", "Samedi", "Dimanche"])))
hw = hw.merge(jours[["date_jour", "evenement"]], on="date_jour")
print("Erreur moyenne par jour de la semaine (%) :")
print(hw.groupby("jour", sort=False)["erreur_pct"].mean().round(1).to_string())
print("\nErreur moyenne selon l'evenement (%) :")
print(hw.groupby("evenement")["erreur_pct"].agg(["mean", "size"]).round(1).to_string())

À regarder :
- L'erreur est la plus forte le samedi (≈ 14 %) : c'est le jour le plus gros et le plus variable.
- Dans ces données simulées, les jours d'événement ne sont presque pas plus mal prévus (Pâques ≈ 12 %, Soldes ≈ 11 %, contre ≈ 11 % sans événement). Dans un vrai magasin, ils le seraient souvent davantage, car Holt-Winters ne connaît pas le calendrier commercial.
- Piste d'amélioration pour le rapport : donner les événements au modèle (par exemple avec un modèle SARIMAX ou une régression).

## 7. Exporter les prévisions pour Power BI

On enregistre toutes les prévisions dans un fichier CSV.

In [ ]:
export = resultats.pivot_table(index=["lundi", "date_jour", "reel"], columns="methode",
                               values="prevision").reset_index()
export.columns.name = None
export = export.round(2)
export.to_csv("previsions_7j.csv", index=False, sep=";", decimal=",", encoding="utf-8-sig")
print(f"previsions_7j.csv enregistre : {len(export)} lignes")
export.head()

Le fichier utilise `;` comme séparateur et `,` pour les décimales .

## 8. Synthèse (pour le rapport, section 5.3)
> « Avec une validation glissante sur 32 semaines de 2025, un lissage exponentiel de Holt-Winters à saisonnalité hebdomadaire prévoit le CA des 7 prochains jours avec une erreur moyenne d'environ 11 %, soit environ 20 % de moins que la méthode naïve (« même jour la semaine dernière »). L'objectif du cahier des charges (-15 %) est atteint. »

Limites : données simulées ; les événements commerciaux ne sont pas intégrés ; une seule série (le magasin entier), pas encore par rayon.

